# DSCI 531 Week 2：Altair 可运行演示

这不是 Lab 答案。它只演示 Week 2：**summary 与 spread、counts/proportions、histogram、KDE、joint distributions、EDA 与 missingness**。

从仓库根目录运行：

```sh
uv run --project DSCI_531/assignments/lab2-py/DSCI_531_lab2-py_yz2000 jupyter lab
```

打开 `DSCI_531/notes/week2-altair-demo.ipynb`，按顺序 Restart Kernel and Run All。数据完全内置，不需要网络。

## 0. Setup：内置合成数据

一行是一台设备在一天的一次观测。`signal` 有 4 个缺失值，并人为加入 1 个极端高值，便于观察 summary 会隐藏什么。

In [1]:
import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display

alt.data_transformers.disable_max_rows()

observations = pd.MultiIndex.from_product(
    [range(1, 21), range(1, 13)], names=["device_num", "day"]
).to_frame(index=False)
observations["device"] = observations["device_num"].map(lambda x: f"D{x:02d}")
region_names = np.array(["North", "Central", "South"])
observations["region"] = pd.Categorical(
    region_names[(observations["device_num"] - 1) % 3],
    categories=region_names, ordered=True)
region_load = observations["region"].map(
    {"North": 3, "Central": 0, "South": -2}).astype(float)
region_signal = observations["region"].map(
    {"North": 1.5, "Central": 0, "South": -1}).astype(float)
region_events = observations["region"].map(
    {"North": 2, "Central": 1, "South": 0}).astype(float)
observations["load"] = (
    20 + 1.7 * observations["day"] + region_load
    + ((observations["device_num"] * 17 + observations["day"] * 7) % 11 - 5) / 2)
observations["signal"] = (
    4 + 0.38 * observations["load"]
    + 2 * np.sin(observations["day"] * np.pi / 6) + region_signal
    + ((observations["device_num"] * 5 + observations["day"] * 3) % 7 - 3) * 0.4)
observations["events"] = np.maximum(
    0, np.round(observations["load"] / 6 + region_events
                + ((observations["device_num"] + observations["day"]) % 3 - 1))
).astype(int)
observations.loc[(observations["device_num"] == 7) & (observations["day"] == 8), "signal"] += 12
missing_pairs = {(3, 4), (8, 9), (14, 2), (19, 11)}
missing = [p in missing_pairs for p in zip(observations["device_num"], observations["day"])]
observations.loc[missing, "signal"] = np.nan
observations["alert"] = np.select(
    [observations["signal"].isna(), observations["signal"] >= 22],
    ["missing", "alert"], default="ok")
observations = observations[[
    "device", "device_num", "region", "day",
    "load", "signal", "events", "alert"]]

assert observations.shape == (240, 8)
assert observations["signal"].isna().sum() == 4
display(observations.head(8))
observations.info()

,device,device_num,region,day,load,signal,events,alert
0,D01,1,North,1,23.2,14.516000,7,ok
1,D01,1,North,2,28.4,18.424051,6,ok
2,D01,1,North,3,28.1,16.978000,7,ok
3,D01,1,North,4,27.8,17.796051,8,ok
4,D01,1,North,5,33.0,20.240000,6,ok
5,D01,1,North,6,32.7,17.526000,7,ok
6,D01,1,North,7,32.4,17.612000,8,ok
7,D01,1,North,8,37.6,17.255949,7,ok


<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   device      240 non-null    str     
 1   device_num  240 non-null    int64   
 2   region      240 non-null    category
 3   day         240 non-null    int64   
 4   load        240 non-null    float64 
 5   signal      236 non-null    float64 
 6   events      240 non-null    int64   
 7   alert       240 non-null    str     
dtypes: category(1), float64(2), int64(3), str(2)
memory usage: 14.8 KB


## 1. Central tendency、spread 与 underlying observations

Mean、median、SD、boxplot 都压缩信息。Beyond-whisker point 只是值得检查，不自动等于错误。这里的 spread 描述 observations，不是 mean 的 uncertainty。

In [2]:
summary_base = alt.Chart(observations, width=330, height=250).encode(
    y="region:N", x="signal:Q")
raw_ticks = summary_base.mark_tick(opacity=0.25)
mean_points = summary_base.mark_point(
    color="black", filled=True, size=90).encode(x="mean(signal):Q")
raw_and_mean = (raw_ticks + mean_points).properties(title="原始观测 + mean")
boxplots = summary_base.mark_boxplot(extent=1.5).properties(
    title="Median、IQR、whiskers")
display(raw_and_mean | boxplots)

alt.HConcatChart(...)

## 2. Counts、proportions 与 denominator

Joint count = 一个 `(region, alert)` cell；marginal count = row/column total。Joint proportion 除以 grand total；conditional proportion 除以指定 row/column total。Normalized bars 隐藏 group size。

In [3]:
count_plot = (
    alt.Chart(observations, width=330, height=250).mark_bar()
    .encode(x="region:N", y="count():Q", color="alert:N", xOffset="alert:N")
    .properties(title="Joint counts：region × alert"))
conditional_plot = (
    alt.Chart(observations, width=330, height=250).mark_bar()
    .encode(x="region:N", y=alt.Y("count():Q").stack("normalize"), color="alert:N")
    .properties(title="Conditional proportions：alert within region"))
display(count_plot | conditional_plot)
counts = observations.groupby(["region", "alert"], observed=True).size().rename("n").reset_index()
counts["within_region"] = counts["n"] / counts.groupby("region", observed=True)["n"].transform("sum")
display(counts)

alt.HConcatChart(...)

,region,alert,n,within_region
0,North,alert,4,0.047619
1,North,missing,1,0.011905
2,North,ok,79,0.940476
3,Central,missing,2,0.023810
4,Central,ok,82,0.976190
5,South,missing,1,0.013889
6,South,ok,71,0.986111


## 3. Histogram：bin width 和 origin 都会改变图形

Histogram 先分 bin，再数 interval 中的 observations。不要只看一种 width/origin；检查 peak/gap 是否稳定。

In [4]:
def histogram(step, anchor, title):
    return (
        alt.Chart(observations, width=220, height=220).mark_bar()
        .encode(
            x=alt.X("signal:Q").bin(step=step, anchor=anchor),
            y="count():Q")
        .properties(title=title))

hist_narrow = histogram(1, 0, "step=1; anchor=0")
hist_wide = histogram(4, 0, "step=4; anchor=0")
hist_shifted = histogram(4, 2, "step=4; anchor=2")
display(hist_narrow | hist_wide | hist_shifted)

alt.HConcatChart(...)

## 4. KDE、bandwidth 与 grouped distributions

Normalized KDE 每组总面积为 1；高度是 probability density per x-unit，不是 point probability/count。小 bandwidth 显示细节/噪声，大 bandwidth 更平滑但可能隐藏 modes。光滑曲线不证明 n 大。

In [5]:
def grouped_density(bandwidth, title):
    return (
        alt.Chart(observations, width=330, height=240)
        .transform_density(
            "signal", as_=["signal", "density"],
            groupby=["region"], bandwidth=bandwidth)
        .mark_line()
        .encode(x="signal:Q", y="density:Q", color="region:N")
        .properties(title=title))

display(grouped_density(0.6, "KDE bandwidth=0.6") |
        grouped_density(2.0, "KDE bandwidth=2.0"))

density_facets = (
    alt.Chart(observations, height=90)
    .transform_density("signal", as_=["signal", "density"], groupby=["region"], bandwidth=1.2)
    .mark_area(opacity=0.55)
    .encode(x="signal:Q", y=alt.Y("density:Q").axis(None))
    .facet(row=alt.Row("region:N").title(None), spacing=0)
    .properties(title="Compact grouped densities"))
display(density_facets)

alt.HConcatChart(...)

alt.FacetChart(...)

## 5. Joint distributions：margins 不能重建 combinations

分别知道两个 variables 的 marginal distributions，不能确定每个 cell 的 joint count。Scatter 饱和时换 2-D histogram；同样要改变 x/y bin step 做 sensitivity check。

In [6]:
category_joint = (
    alt.Chart(observations, width=300, height=220).mark_circle()
    .encode(x="alert:N", y="region:N", size="count():Q", color="count():Q")
    .properties(title="Categorical joint counts"))
joint_points = (
    alt.Chart(observations, width=300, height=220).mark_point(opacity=0.25)
    .encode(x="load:Q", y="signal:Q")
    .properties(title="Scatter：overlap 可能掩盖密度"))
joint_bins = (
    alt.Chart(observations, width=300, height=220).mark_rect()
    .encode(
        x=alt.X("load:Q").bin(step=3),
        y=alt.Y("signal:Q").bin(step=2), color="count():Q")
    .properties(title="2-D histogram：joint count"))
display(category_joint | joint_points | joint_bins)

alt.HConcatChart(...)

## 6. EDA：distribution、relationship、missingness

Reasoning trail：**expectation → observation → competing explanations → discriminating check → narrow conclusion + unresolved**。Valid zero 不等于 missing；极端值也不自动等于错误。

In [7]:
measurements = ["load", "signal", "events"]
distribution_gallery = (
    alt.Chart(observations).mark_bar()
    .encode(
        x=alt.X(alt.repeat(), type="quantitative", bin=alt.Bin(maxbins=20)),
        y=alt.Y("count():Q").title("Recorded rows"))
    .properties(width=210, height=180)
    .repeat(measurements, columns=3))

recorded_share = (
    observations.assign(recorded=observations["signal"].notna())
    .groupby(["day", "region"], observed=True, as_index=False)
    .agg(recorded_share=("recorded", "mean")))
missing_plot = (
    alt.Chart(recorded_share, width=650, height=130).mark_rect()
    .encode(
        x=alt.X("day:O").axis(labelAngle=0), y="region:N",
        color=alt.Color("recorded_share:Q").scale(domain=[0, 1]),
        tooltip=["day:O", "region:N", alt.Tooltip("recorded_share:Q", format=".1%")])
    .properties(title="Recorded-share heatmap"))
outcome_gallery = (
    alt.Chart(observations).mark_point(size=18, opacity=0.25)
    .encode(
        x=alt.X(alt.repeat("column"), type="quantitative").scale(zero=False),
        y=alt.Y("signal:Q").scale(zero=False), color="region:N")
    .properties(width=300, height=230)
    .repeat(column=["load", "events"]))

display(distribution_gallery)
display(missing_plot)
display(outcome_gallery)

alt.RepeatChart(...)

alt.Chart(...)

alt.RepeatChart(...)

## 7. 建议你亲手修改

1. 在 summary 图上加入/移除 raw observations，观察 summary 隐藏什么。
2. 把 normalized bar 改回 raw count，明确 denominator。
3. 修改 histogram 的 `step` 和 `anchor`。
4. 修改 KDE 的 `bandwidth`，并比较 group n。
5. 修改 2-D histogram 的 x/y bin step。
6. 把 missing value 填成 0，观察图怎样改变；解释为什么这不是中立处理。